# Simulation Optimization

Combining simulation with optimization to solve complex stochastic problems where analytical solutions are difficult or impossible.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import differential_evolution

sns.set(style="whitegrid")
np.random.seed(42)

## Simulation Model: (s, S) Inventory Policy

In [ ]:
class InventorySimulator:
    def __init__(self, lead_time=3, holding_cost=1.0, shortage_cost=15.0, review_period=1):
        self.lead_time = lead_time
        self.holding_cost = holding_cost
        self.shortage_cost = shortage_cost
        self.review_period = review_period
    
    def simulate(self, s, S, num_days=365, num_replications=50):
        total_costs = []
        
        for _ in range(num_replications):
            inventory = S
            daily_cost = 0
            pending_orders = []
            
            for day in range(num_days):
                # Receive orders due today
                if pending_orders and pending_orders[0][0] == day:
                    inventory += pending_orders.pop(0)[1]
                
                # Generate stochastic demand
                demand = np.random.poisson(12)
                
                # Fulfill demand
                sales = min(inventory, demand)
                shortage = demand - sales
                inventory -= sales
                
                daily_cost += self.holding_cost * max(inventory, 0) + self.shortage_cost * shortage
                
                # Review and order (periodic review (s, S) policy)
                if day % self.review_period == 0:
                    if inventory < s:
                        order_qty = S - inventory
                        pending_orders.append((day + self.lead_time, order_qty))
            
            total_costs.append(daily_cost)
        
        return np.mean(total_costs)

sim = InventorySimulator()
print("Simulation model ready")

## Optimize (s, S) Parameters using Simulation

In [ ]:
def objective(x):
    s, S = int(x[0]), int(x[1])
    if S <= s:
        return 1e6  # invalid solution penalty
    return sim.simulate(s, S, num_replications=30)

# Bounds: s from 0 to 40, S from s+1 to 80
bounds = [(0, 40), (10, 80)]

result = differential_evolution(objective, bounds, workers=1, tol=1.0, popsize=15)

print("Optimization completed")
print(f"Best (s, S) policy found: s = {int(result.x[0])}, S = {int(result.x[1])}")
print(f"Estimated average daily cost: {result.fun:.2f}")

In [ ]:
# Evaluate performance of the optimized policy
best_s = int(result.x[0])
best_S = int(result.x[1])

costs = [sim.simulate(best_s, best_S, num_replications=100) for _ in range(5)]
print(f"Final average cost with optimized policy: {np.mean(costs):.2f} ± {np.std(costs):.2f}")

## Visualization of Simulation Results

In [ ]:
# Run multiple replications for visualization
replications = 200
daily_costs = []

for _ in range(replications):
    cost = sim.simulate(best_s, best_S, num_days=365, num_replications=1)
    daily_costs.append(cost / 365)

plt.figure(figsize=(10, 6))
plt.hist(daily_costs, bins=40, alpha=0.7, color='steelblue', edgecolor='black')
plt.axvline(np.mean(daily_costs), color='red', linestyle='--', label=f'Mean = {np.mean(daily_costs):.2f}')
plt.title('Distribution of Daily Costs under Optimized (s, S) Policy')
plt.xlabel('Average Daily Cost')
plt.ylabel('Frequency')
plt.legend()
plt.show()

## Exercises

- Extend the simulator to include multiple items or capacitated warehouses.
- Replace differential_evolution with Bayesian Optimization or Optuna.
- Integrate a machine learning surrogate model to speed up the simulation evaluations.
- Apply simulation optimization to ambulance location or call center staffing problems.

The next chapter will cover **Prescriptive Analytics** and end-to-end OR + ML pipelines.